In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import gc
import os
from tqdm import tqdm
import numpy as np

warnings.simplefilter('ignore')
%matplotlib inline

pd.set_option('max_columns', None)
pd.set_option('max_rows', None)

In [ ]:
seed = 2020

In [ ]:
# !pip install xgboost

In [ ]:
df_feature = pd.read_pickle('data/feature.pkl')

In [ ]:
df_feature.head()

In [ ]:
for f in df_feature.select_dtypes('object').columns:
    lbl = LabelEncoder()
    df_feature[f] = lbl.fit_transform(df_feature[f].astype(str))

In [ ]:
df_train = df_feature[df_feature.isDefault.notna()].copy()
df_test = df_feature[df_feature.isDefault.isna()].copy()

df_train.shape, df_test.shape

In [ ]:
ycol = 'isDefault'
feature_names = list(
    filter(lambda x: x not in [ycol, 'id'], df_train.columns))

model = xgb.XGBClassifier(max_depth=6,
                          learning_rate=0.05,
                          n_estimators=10000,
                          subsample=0.8,
                          reg_alpha=10,
                          reg_lambda=12,
#                           tree_method='gpu_hist',
                          random_state=seed)

df_oof = df_train[['id', ycol]].copy()
df_oof['prob'] = 0
prediction = df_test[['id']]
prediction['prob'] = 0
df_importance_list = []

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
for fold_id, (trn_idx, val_idx) in enumerate(
        kfold.split(df_train[feature_names], df_train[ycol])):
    X_train = df_train.iloc[trn_idx][feature_names]
    Y_train = df_train.iloc[trn_idx][ycol]

    X_val = df_train.iloc[val_idx][feature_names]
    Y_val = df_train.iloc[val_idx][ycol]

    print('\nFold_{} Training ================================\n'.format(
        fold_id + 1))

    xgb_model = model.fit(X_train,
                          Y_train,
                          eval_set=[(X_train, Y_train), (X_val, Y_val)],
                          verbose=100,
                          eval_metric='auc',
                          early_stopping_rounds=50)

    pred_val = xgb_model.predict_proba(X_val)[:, 1]
    df_oof.loc[val_idx, 'prob'] = pred_val

    pred_test = xgb_model.predict_proba(df_test[feature_names])[:, 1]
    prediction['prob'] += pred_test / kfold.n_splits

    df_importance = pd.DataFrame({
        'column': feature_names,
        'importance': xgb_model.feature_importances_,
    })
    df_importance_list.append(df_importance)

    del xgb_model, pred_val, pred_test, X_train, Y_train, X_val, Y_val
    gc.collect()

In [ ]:
df_importance = pd.concat(df_importance_list)
df_importance = df_importance.groupby([
    'column'
])['importance'].agg('mean').sort_values(ascending=False).reset_index()
df_importance

In [ ]:
auc = roc_auc_score(df_oof[ycol], df_oof['prob'])
print('auc:', auc)

In [ ]:
os.makedirs('sub', exist_ok=True)

sub = prediction[['id', 'prob']].copy()
sub.columns = ['id', 'isDefault']
sub.to_csv('sub/jiedai_{}.csv'.format(auc), index=False)

In [ ]:
os.makedirs('prob', exist_ok=True)

prediction.to_csv('prob/sub_xgb.csv', index=False)
df_oof[['id', 'prob', ycol]].to_csv('prob/oof_xgb.csv', index=False)

prediction.to_csv('prob/sub_xgb_{}.csv'.format(auc), index=False)
df_oof[['id', 'prob', ycol]].to_csv(
    'prob/oof_xgb_{}.csv'.format(auc), index=False)

In [ ]:
df_oof.head()

In [ ]:
prediction.head()